In [48]:
import time

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph, START, END, MessagesState

FIRST_STAGE = "first_stage"
SECOND_STAGE = "second_stage"
FINAL_STAGE = "final_stage"

In [49]:
process_status = {"is_interruption": False}

In [50]:
class MsgState(MessagesState):
    user_request:str
    response:str

In [52]:
def process_first_stage(state: MsgState):
    print("\nFirst Stage Process Begins...")
    state["user_request"] = state["user_request"] + "- Stage One"
    state["response"] = f"{state["user_request"]} - Clearance from First stage process"
    print(state)
    return state

def process_second_stage(state: MsgState):
    print("\nSecond Stage Process Begins...")
    state["user_request"] = state["user_request"] + "- Stage Two"
    time.sleep(3)
    if process_status["is_interruption"]:
        process_status["is_interruption"] = False
        print("\nProcess Interrupted")
        raise ValueError("interrupted due to raw material mix")
    state["response"] = f"{state["user_request"]} - Clearance from Second stage process"
    print(state)
    return state

def process_final_stage(state:MsgState):
    print("\nFinal Stage Process Begins...")
    state["user_request"] = state["user_request"] + "- Process Completed"
    state["response"] = f"{state["user_request"]} - Final Clearance -- Product is ready"
    print(state)
    return state

In [53]:
wf_graph = StateGraph(MsgState)
wf_graph.add_node(FIRST_STAGE, process_first_stage)
wf_graph.add_node(SECOND_STAGE, process_second_stage)
wf_graph.add_node(FINAL_STAGE, process_final_stage)

wf_graph.add_edge(START, FIRST_STAGE)
wf_graph.add_edge(FIRST_STAGE, SECOND_STAGE)
wf_graph.add_edge(SECOND_STAGE, FINAL_STAGE)
wf_graph.add_edge(FINAL_STAGE, END)

in_memory_checkpoint = InMemorySaver()
graph = wf_graph.compile(in_memory_checkpoint)

### Execute with Interruption / Exception

In [62]:
configs = {"configurable":{"thread_id":"user-1"}}
try:
    #response = graph.invoke({"messages":[{"user_request":"Refine Oil"}]}, configs)
    #response = graph.invoke({"messages":[{"role":"user", "content":"Refine Oil"}]}, configs)
    response = graph.invoke({"user_request":"Refine Oil"}, config=configs)
    print(response)
except KeyboardInterrupt as ke:
    print(f"Power cut interrupt : {ke}")
except Exception as e:
    print(f"Interruption : {e}")


First Stage Process Begins...
{'messages': [], 'user_request': 'Refine Oil- Stage One', 'response': 'Refine Oil- Stage One - Clearance from First stage process'}

Second Stage Process Begins...
Power cut interrupt : 


### Execute - Post Interruption / Exception

In [63]:
configs = {"configurable":{"thread_id":"user-1"}}
try:
    response = graph.invoke(None, config=configs)
    print(response)
except Exception as e:
    print(f"Interruption : {e}")


Second Stage Process Begins...
{'messages': [], 'user_request': 'Refine Oil- Stage One- Stage Two', 'response': 'Refine Oil- Stage One- Stage Two - Clearance from Second stage process'}

Final Stage Process Begins...
{'messages': [], 'user_request': 'Refine Oil- Stage One- Stage Two- Process Completed', 'response': 'Refine Oil- Stage One- Stage Two- Process Completed - Final Clearance -- Product is ready'}
{'messages': [], 'user_request': 'Refine Oil- Stage One- Stage Two- Process Completed', 'response': 'Refine Oil- Stage One- Stage Two- Process Completed - Final Clearance -- Product is ready'}


In [57]:
configs1 = {"configurable":{"thread_id":"user-2"}}
response = graph.invoke({"user_request":"Refine Palm Oil"}, config=configs1)
print(response)


First Stage Process Begins...
{'messages': [], 'user_request': 'Refine Palm Oil- Stage One', 'response': 'Refine Palm Oil- Stage One - Clearance from First stage process'}

Second Stage Process Begins...
{'messages': [], 'user_request': 'Refine Palm Oil- Stage One- Stage Two', 'response': 'Refine Palm Oil- Stage One- Stage Two - Clearance from Second stage process'}

Final Stage Process Begins...
{'messages': [], 'user_request': 'Refine Palm Oil- Stage One- Stage Two- Process Completed', 'response': 'Refine Palm Oil- Stage One- Stage Two- Process Completed - Final Clearance -- Product is ready'}
{'messages': [], 'user_request': 'Refine Palm Oil- Stage One- Stage Two- Process Completed', 'response': 'Refine Palm Oil- Stage One- Stage Two- Process Completed - Final Clearance -- Product is ready'}


In [60]:
#print(in_memory_checkpoint.get(config=configs1))
in_memory_checkpoint.get_tuple(config=configs)

CheckpointTuple(config={'configurable': {'thread_id': 'user-1', 'checkpoint_ns': '', 'checkpoint_id': '1f17eacb-8592-6794-8006-ea831c87deef'}}, checkpoint={'v': 4, 'ts': '2026-07-13T11:19:31.752114+00:00', 'id': '1f17eacb-8592-6794-8006-ea831c87deef', 'channel_versions': {'__start__': '00000000000000000000000000000005.0.9460446401985206', 'user_request': '00000000000000000000000000000008.0.6595758964755031', 'branch:to:first_stage': '00000000000000000000000000000006.0.47991540912752917', 'messages': '00000000000000000000000000000008.0.6595758964755031', 'response': '00000000000000000000000000000008.0.6595758964755031', 'branch:to:second_stage': '00000000000000000000000000000007.0.7875614178588719', 'branch:to:final_stage': '00000000000000000000000000000008.0.6595758964755031'}, 'versions_seen': {'__input__': {}, '__start__': {'__start__': '00000000000000000000000000000004.0.08508863073764272'}, 'first_stage': {'branch:to:first_stage': '00000000000000000000000000000005.0.946044640198520